# Exploração da Ingestão — Membro 3

Notebook para rodar os extractors de `src/extractors/` por partes e inspecionar os resultados antes de plugar nas DAGs do Airflow.

- **Seção 1**: API REST do Ceará Transparente (contratos)
- **Seção 2**: PostgreSQL de origem (`empenhos`, `ordem_bancaria_orcamentaria`, `unidade_gestora`)

> Antes de rodar: copie `../.env.example` para `../.env` e ajuste `SOURCE_POSTGRES_URL` com as credenciais reais da infra do time. As células da Seção 1 (API pública) funcionam sem configuração adicional.

In [2]:
import sys
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd

# Dentro do Jupyter não existe __file__ — a raiz do projeto é o pai do
# diretório de trabalho atual (o notebook roda a partir de notebooks/).
project_root = Path.cwd().parent

# Adiciona ao sys.path se ainda não estiver lá
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Carrega o arquivo .env localizado na raiz do projeto
env_path = project_root / ".env"
load_dotenv(dotenv_path=env_path)

print(f"Project root: {project_root}")

Project root: c:\Projeto T03\projeto-final


## 1. API REST — Ceará Transparente (contratos)

### 1.1 Sanity check — uma única página

Antes de iterar tudo, vale olhar a resposta bruta de uma página só para conferir o formato (`summary.total_pages`, campos de `data`).

In [3]:
import requests
from src.extractors import api_extractor

DATA_INICIO = "2026-01-01"
DATA_FIM = "2026-01-31"

session = requests.Session()
primeira_pagina = api_extractor._request_page(session, page=1, data_assinatura_inicio=DATA_INICIO, data_assinatura_fim=DATA_FIM)

print("Chaves do payload:", list(primeira_pagina.keys()))
print("Summary:", primeira_pagina.get("summary"))
print("Registros na página 1:", len(primeira_pagina.get("data") or primeira_pagina.get("results") or []))

Chaves do payload: ['sumary', 'data']
Summary: None
Registros na página 1: 100


In [4]:
registros = (primeira_pagina.get("data") or primeira_pagina.get("results") or [])
pd.DataFrame(registros).head()

,id,cod_concedente,cod_financiador,cod_gestora,cod_orgao,cod_secretaria,descricao_modalidade,descricao_objeto,descricao_tipo,descricao_url,...,gestor_contrato,data_finalizacao_prestacao_contas,sequential,emergency,law,has_non_profit_transfer,nome_fiscal,emenda_parlamentar,ano_emenda_parlamentar,codigo_emenda_parlamentar
0,643345,080701,1414,080901,08200007,08000000,PREGÃO ELETRÔNICO,CONTRATO PARA COBERTURA DE CUSTOS EM ASSISTÊNC...,CONTRATO,20260211.1414246.Integra.CONTRATO.pdf,...,JOSE TUPINAMBA CAVALCANTE DE ALMEIDA,0001-01-01,3008,False,None,None,None,None,None,None
1,643348,080701,1012871,080901,08200007,08000000,PREGÃO ELETRÔNICO,CONTRATO DE ADESÃO AO PLANO DE CONTRIBUIÇÃO DE...,CONTRATO,20260211.1414253.Integra.CONTRATO.pdf,...,JOSE TUPINAMBA CAVALCANTE DE ALMEIDA,0001-01-01,3011,False,13303,None,GARDÊNIA GOERSCH ANDRADE PARENTE,None,None,None
2,643346,080701,291746,080901,08200007,08000000,PREGÃO ELETRÔNICO,CONCESSÃO DE PLANO COLETIVO EMPRESARIAL DO TI...,CONTRATO,20260211.1414240.Integra.CONTRATO.pdf,...,JOSE TUPINAMBA CAVALCANTE DE ALMEIDA,0001-01-01,3009,False,None,None,None,None,None,None
3,643349,080701,3215,080901,08200007,08000000,PREGÃO ELETRÔNICO,CESSÃO DE UTILIZAÇÃO DOS CARTOES ELETRONICOS I...,CONTRATO,20260211.1414256.Integra.CONTRATO.pdf,...,JOSE TUPINAMBA CAVALCANTE DE ALMEIDA,0001-01-01,3012,False,13303,None,GARDÊNIA GOERSCH ANDRADE PARENTE,-,None,None
4,643343,080701,68913,080901,08200007,08000000,PREGÃO ELETRÔNICO,CONCESSÃO DE PLANO COLETIVO EMPRESARIAL DO TI...,CONTRATO,20260211.1414205.Integra.CONTRATO.pdf,...,JOSE TUPINAMBA CAVALCANTE DE ALMEIDA,0001-01-01,3006,False,None,None,None,None,None,None


### 1.2 Paginação completa (sem gravar ainda)

Itera todas as páginas do intervalo usando `fetch_contratos`, mostrando o progresso página a página — a mesma função usada de verdade na extração, só que aqui coletamos tudo em memória para inspecionar.

In [6]:
todas_as_paginas = []

for page, records in api_extractor.fetch_contratos(DATA_INICIO, DATA_FIM):
    print(f"Página {page}: {len(records)} registros")
    todas_as_paginas.extend(records)

df_contratos = pd.DataFrame(todas_as_paginas)
print(f"\nTotal de registros: {len(df_contratos)}")
df_contratos.head()

Página 1: 100 registros
Página 2: 100 registros
Página 3: 100 registros
Página 4: 100 registros
Página 5: 100 registros
Página 6: 100 registros
Página 7: 100 registros
Página 8: 36 registros

Total de registros: 736


,id,cod_concedente,cod_financiador,cod_gestora,cod_orgao,cod_secretaria,descricao_modalidade,descricao_objeto,descricao_tipo,descricao_url,...,gestor_contrato,data_finalizacao_prestacao_contas,sequential,emergency,law,has_non_profit_transfer,nome_fiscal,emenda_parlamentar,ano_emenda_parlamentar,codigo_emenda_parlamentar
0,643345,080701,1414,080901,08200007,08000000,PREGÃO ELETRÔNICO,CONTRATO PARA COBERTURA DE CUSTOS EM ASSISTÊNC...,CONTRATO,20260211.1414246.Integra.CONTRATO.pdf,...,JOSE TUPINAMBA CAVALCANTE DE ALMEIDA,0001-01-01,3008,False,None,None,None,None,None,None
1,643348,080701,1012871,080901,08200007,08000000,PREGÃO ELETRÔNICO,CONTRATO DE ADESÃO AO PLANO DE CONTRIBUIÇÃO DE...,CONTRATO,20260211.1414253.Integra.CONTRATO.pdf,...,JOSE TUPINAMBA CAVALCANTE DE ALMEIDA,0001-01-01,3011,False,13303,None,GARDÊNIA GOERSCH ANDRADE PARENTE,None,None,None
2,643346,080701,291746,080901,08200007,08000000,PREGÃO ELETRÔNICO,CONCESSÃO DE PLANO COLETIVO EMPRESARIAL DO TI...,CONTRATO,20260211.1414240.Integra.CONTRATO.pdf,...,JOSE TUPINAMBA CAVALCANTE DE ALMEIDA,0001-01-01,3009,False,None,None,None,None,None,None
3,643349,080701,3215,080901,08200007,08000000,PREGÃO ELETRÔNICO,CESSÃO DE UTILIZAÇÃO DOS CARTOES ELETRONICOS I...,CONTRATO,20260211.1414256.Integra.CONTRATO.pdf,...,JOSE TUPINAMBA CAVALCANTE DE ALMEIDA,0001-01-01,3012,False,13303,None,GARDÊNIA GOERSCH ANDRADE PARENTE,-,None,None
4,643343,080701,68913,080901,08200007,08000000,PREGÃO ELETRÔNICO,CONCESSÃO DE PLANO COLETIVO EMPRESARIAL DO TI...,CONTRATO,20260211.1414205.Integra.CONTRATO.pdf,...,JOSE TUPINAMBA CAVALCANTE DE ALMEIDA,0001-01-01,3006,False,None,None,None,None,None,None


In [ ]:
df_contratos.info()

### 1.3 Extração completa gravando na Bronze

Agora sim usando `extract_and_save`, que grava cada página como JSON no destino configurado em `.env` (`BRONZE_STORAGE_BACKEND=local` ou `hdfs`) e retorna só os metadados (contagens) — o mesmo formato que a task do Airflow vai receber via XCom.

In [7]:
resultado_api = api_extractor.extract_and_save(DATA_INICIO, DATA_FIM)
print(resultado_api)

{'total_pages': 8, 'total_records': 736, 'run_date': '2026-07-19', 'max_data_assinatura': '2026-01-31'}


## 2. PostgreSQL de origem

Requer `SOURCE_POSTGRES_URL` configurada no `.env` apontando para a infra real do time.

### 2.1 Teste de conexão

In [8]:
from sqlalchemy import create_engine, text
from src.extractors import postgres_extractor

engine = create_engine(postgres_extractor.SOURCE_DB_URL)

with engine.connect() as conn:
    versao = conn.execute(text("SELECT version()")).scalar()
print("Conectado com sucesso:", versao)

Conectado com sucesso: PostgreSQL 18.1 (Debian 18.1-1.pgdg13+2) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit


### 2.2 Extração tabela a tabela (sem gravar ainda)

Roda `extract_table` para cada tabela do escopo, mostrando shape e uma amostra — bom momento para conferir se os nomes de coluna de data em `TABLE_DATE_COLUMNS` (`data_empenho`, `data_pagamento`) batem com o schema real.

In [9]:
DATA_INICIO_PG = "2026-01-01"
DATA_FIM_PG = "2026-01-10"

dataframes = {}
for tabela in postgres_extractor.TABLE_DATE_COLUMNS:
    df = postgres_extractor.extract_table(tabela, DATA_INICIO_PG, DATA_FIM_PG, engine=engine)
    dataframes[tabela] = df
    print(f"{tabela}: shape={df.shape}")

empenhos: shape=(0, 30)
ordem_bancaria_orcamentaria: shape=(186, 42)
unidade_gestora: shape=(5011, 16)


In [ ]:
dataframes["empenhos"].head()

In [ ]:
dataframes["ordem_bancaria_orcamentaria"].head()

In [ ]:
dataframes["unidade_gestora"].head()

### 2.3 Extração completa gravando na Bronze

In [ ]:
resultado_pg = postgres_extractor.extract_and_save(DATA_INICIO_PG, DATA_FIM_PG, engine=engine)
print(resultado_pg)

In [ ]:
engine.dispose()

## Próximos passos

- Conferir se `TABLE_DATE_COLUMNS` em `postgres_extractor.py` bate com o schema real (colunas de data de `empenhos` e `ordem_bancaria_orcamentaria`).
- Confirmar `BRONZE_BASE_PATH` / `HDFS_WEBHDFS_URL` com a infra que o time já tem rodando.
- Fase 2: enriquecimento cruzando `empenhos` × `unidade_gestora`.